# M5 — CV/Transcript NER Extractor (BERT-base-uncased)
Target: **Entity F1 >= 0.82** (on the gold test split). See `docs/phases/MENTORA_Phase4_Model_Training.md` §5.

**Known limitation carried over from Phase 3** (`mentora_data/SOURCES.md`):
the gold set (`gold_val_test.jsonl`) is auto-tagged, not yet hand-corrected.
Training on silver / evaluating on gold is still the right discipline (it's
the best split available), but until someone manually reviews
`mentora_data/labeled/m5/gold_review.csv`, the reported F1 reflects
auto-tagger quality on both sides, not auto-tagger-vs-human-truth. Schema
(`tokens`, `ner_tags` per line) matches the plan doc exactly — no adaptation
needed here.

## Setup — mount Drive, confirm GPU, install deps
Same first 3 cells in every Mentora training notebook. See `docs/phases/MENTORA_Phase4_Model_Training.md` §0.

In [21]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thu Oct  8 05:34:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

In [22]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

In [27]:
!pip install -q -U pip setuptools wheel
!pip install -q transformers accelerate evaluate
!pip install -q seqeval --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 40.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ...

In [28]:
!pip install -q "setuptools<82" jedi
!pip install -q seqeval --no-build-isolation

## Load gold (val/test) and silver (train) sets

In [29]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

silver = load_jsonl(f'{DATA}/labeled/m5/silver_train.jsonl')
gold = load_jsonl(f'{DATA}/labeled/m5/gold_val_test.jsonl')
print(f"{len(silver)} silver (train), {len(gold)} gold (val+test)")

all_tags = sorted({tag for row in silver + gold for tag in row['ner_tags']})
tag2id = {t: i for i, t in enumerate(all_tags)}
id2tag = {i: t for t, i in tag2id.items()}
print("tags:", all_tags)

import random
random.seed(42)
random.shuffle(gold)
val, test = gold[:len(gold)//2], gold[len(gold)//2:]

train_ds = Dataset.from_list(silver)
val_ds = Dataset.from_list(val)
test_ds = Dataset.from_list(test)

812 silver (train), 150 gold (val+test)
tags: ['B-CERT', 'B-DEGREE', 'B-GPA', 'B-SKILL', 'I-CERT', 'I-SKILL', 'O']


## Tokenize with label alignment (sub-token labels for BERT's wordpiece tokenizer)

In [30]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_and_align(batch):
    tokenized = tokenizer(batch['tokens'], truncation=True, is_split_into_words=True, padding='max_length', max_length=256)
    all_labels = []
    for i, tags in enumerate(batch['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids, prev_word = [], None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(tag2id[tags[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

train_ds = train_ds.map(tokenize_and_align, batched=True)
val_ds = val_ds.map(tokenize_and_align, batched=True)
test_ds = test_ds.map(tokenize_and_align, batched=True)

Map:   0%|          | 0/812 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

## Model, metrics (entity-level F1 via seqeval), training

In [33]:
# !pip install -q --no-cache-dir transformers accelerate evaluate seqeval

from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

import torch

model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-uncased', num_labels=len(all_tags), id2label=id2tag, label2id=tag2id, torch_dtype=torch.float32
)
# seqeval = evaluate.load("seqeval")

from seqeval.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(preds, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                p_seq.append(id2tag[int(p)]); l_seq.append(id2tag[int(l)])
        true_preds.append(p_seq); true_labels.append(l_seq)
    return {"entity_f1": f1_score(true_labels, true_preds)}

OUT_DIR = f'{MODELS}/model5_cv_ner'
args = TrainingArguments(
    output_dir=OUT_DIR, per_device_train_batch_size=8, num_train_epochs=4,
    save_strategy="epoch", eval_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="entity_f1", fp16=True,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

import glob
existing = sorted(glob.glob(f'{OUT_DIR}/checkpoint-*'))
trainer.train(resume_from_checkpoint=existing[-1] if existing else None)

print("Test set (gold, held out):", trainer.evaluate(test_ds))
trainer.save_model(f'{OUT_DIR}/final')
tokenizer.save_pretrained(f'{OUT_DIR}/final')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

Epoch,Training Loss,Validation Loss,Entity F1
1,No log,0.036771,0.919367
2,No log,0.019128,0.959879
3,No log,0.021653,0.954307
4,No log,0.024140,0.957958


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Training Loss,Validation Loss,Epoch,Entity F1
No log,0.030778,4,0.949690


Test set (gold, held out): {'eval_loss': 0.030777644366025925, 'eval_entity_f1': 0.9496898690558235}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/mentora_models/model5_cv_ner/final/tokenizer_config.json',
 '/content/drive/MyDrive/mentora_models/model5_cv_ner/final/tokenizer.json')

## If you miss the target (Entity F1 < 0.82)
- Check per-entity-type F1 — CERT usually drags the average down (smallest
  auto-tagger vocabulary in Phase 3)
- Silver-label noise is the other usual culprit — tighten the PhraseMatcher
  vocabulary and regenerate the silver set if spot-checks show high error
- **Most importantly: do the manual gold-set review first** (see the note at
  the top) — training harder against noisy gold labels has a low ceiling